# COVID-19 Mortality Prediction — Machine Learning Pipeline

This notebook builds an **end-to-end ML classification pipeline** to predict patient mortality from COVID-19 based on clinical and demographic features.

**Goal:** Given a patient's medical profile (age, pre-existing conditions, treatment type, etc.), predict whether the outcome is **death** (1) or **survival** (0).

**Pipeline steps:**
1. Load and clean the data
2. Prepare features and target
3. Train/test split
4. Handle class imbalance
5. Train models (Logistic Regression, Random Forest)
6. Evaluate and compare models
7. Analyze feature importance

> **Note:** This is an educational ML project. The models are not intended for clinical decision-making.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load the Raw Data

We load the original COVID-19 dataset collected from Mexican health records. Each row represents one patient, and columns capture demographics, pre-existing conditions, and treatment details.

In [ ]:
df = pd.read_csv('./data/Covid_Data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Data Cleaning

The raw data uses a special encoding:
- **1** = Yes / True
- **2** = No / False
- **97, 98, 99** = Unknown / Not applicable

We convert this to standard binary (1/0) and treat unknowns as missing values. We also create the **DEATH** target column from `DATE_DIED`.

In [ ]:
df.rename(columns={
    'SEX': 'GENDER',
    'MEDICAL_UNIT': 'MEDICAL_TYPE',
    'PATIENT_TYPE': 'TREATMENT_TYPE',
    'CLASIFFICATION_FINAL': 'COVID_RES'
}, inplace=True)

df['DEATH'] = (df['DATE_DIED'] != '9999-99-99').astype(int)
df.drop(columns='DATE_DIED', inplace=True)

print(f'Target distribution:')
print(df['DEATH'].value_counts())
print(f'\nDeath rate: {df["DEATH"].mean():.2%}')

In [ ]:
def fix_unknown(df, columns):
    """Convert 1->1, 2->0, everything else->NaN for boolean-coded columns."""
    for col in columns:
        df[col] = df[col].map({1: 1, 2: 0})
    return df

bool_cols = [
    'INTUBED', 'PNEUMONIA', 'GENDER', 'PREGNANT',
    'DIABETES', 'COPD', 'ASTHMA', 'INMSUPR',
    'HIPERTENSION', 'OTHER_DISEASE', 'CARDIOVASCULAR',
    'OBESITY', 'RENAL_CHRONIC', 'TOBACCO', 'ICU'
]

fix_unknown(df, bool_cols)

df['COVID_RES'] = df['COVID_RES'].apply(lambda x: 0 if x >= 4 else x)

### 3.1 Handle Missing Values

Some columns have a large proportion of missing values after cleaning (e.g., `ICU`, `INTUBED`, `PREGNANT`). We drop columns where more than 40% of values are missing, then drop remaining rows with any NaN.

In [ ]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
print('Missing value percentage per column:\n')
print(missing_pct[missing_pct > 0].apply(lambda x: f'{x:.1%}'))

In [ ]:
high_missing = missing_pct[missing_pct > 0.40].index.tolist()
print(f'Dropping columns with >40% missing: {high_missing}')
df.drop(columns=high_missing, inplace=True)

rows_before = len(df)
df.dropna(inplace=True)
rows_after = len(df)
print(f'Dropped {rows_before - rows_after:,} rows with remaining NaN values')
print(f'Final dataset shape: {df.shape}')

In [ ]:
df.info()

## 4. Prepare Features and Target

We separate the dataset into:
- **X** — all feature columns (everything except DEATH)
- **y** — the target column (DEATH: 0 = survived, 1 = died)

In [ ]:
X = df.drop(columns='DEATH')
y = df['DEATH']

print(f'Features: {X.shape[1]} columns')
print(f'Samples:  {X.shape[0]:,}')
print(f'\nTarget distribution:')
print(y.value_counts())
print(f'\nClass balance — Death rate: {y.mean():.2%}')

## 5. Train/Test Split

We split the data into 80% training and 20% testing sets. We use `stratify=y` to preserve the class distribution in both sets, which is important because our target is imbalanced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTraining death rate: {y_train.mean():.2%}')
print(f'Test death rate:     {y_test.mean():.2%}')

### 5.1 Feature Scaling

Logistic Regression benefits from standardized features (mean=0, std=1). We fit the scaler on the training set only to prevent data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Handle Class Imbalance

The death rate is only ~7%, meaning the dataset is **heavily imbalanced**. A naive model that always predicts "survived" would achieve ~93% accuracy but miss every death.

Since this is a **medical-risk classification**, missing a true death case (false negative) is far worse than a false alarm (false positive). We need to optimize for **recall** — catching as many death cases as possible.

**Strategy:** We use `class_weight='balanced'` in both models, which automatically gives more importance to the minority class (deaths) during training.

## 7. Model Training

We train two models and compare their performance:
1. **Logistic Regression** — a simple, interpretable baseline
2. **Random Forest** — a more powerful ensemble model

### 7.1 Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression — Classification Report:\n')
print(classification_report(y_test, lr_pred, target_names=['Survived', 'Died']))

### 7.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print('Random Forest — Classification Report:\n')
print(classification_report(y_test, rf_pred, target_names=['Survived', 'Died']))

## 8. Model Evaluation & Comparison

We compare both models across multiple metrics. For a medical-risk problem like this, **recall** and **F1-score** on the death class are the most important metrics — we want to catch as many true death cases as possible.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    """Calculate all evaluation metrics for a model."""
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, lr_pred, lr_prob),
    evaluate_model('Random Forest', y_test, rf_pred, rf_prob)
])

results = results.set_index('Model')
results.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen')

### 8.1 Model Comparison — Bar Chart

Visual comparison of key metrics across both models.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

results.T.plot(kind='bar', ax=ax, rot=0, colormap='Set2', edgecolor='black')

ax.set_title('Model Comparison — Evaluation Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(title='Model', loc='lower right')

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.show()

### 8.2 Confusion Matrices

The confusion matrix shows how many predictions fall into each category:
- **True Negative (TN):** Correctly predicted survival
- **False Positive (FP):** Predicted death but patient survived
- **False Negative (FN):** Predicted survival but patient died ← *the most dangerous error*
- **True Positive (TP):** Correctly predicted death

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name, y_pred in zip(axes, ['Logistic Regression', 'Random Forest'], [lr_pred, rf_pred]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt=',d', cmap='Blues',
        xticklabels=['Survived', 'Died'],
        yticklabels=['Survived', 'Died'],
        ax=ax
    )
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 8.3 ROC Curves

The ROC curve shows the trade-off between the true positive rate (recall) and false positive rate at different classification thresholds. A higher AUC indicates better overall discrimination ability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_test, lr_prob, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, rf_prob, name='Random Forest', ax=ax)

ax.plot([0, 1], [0, 1], 'k--', label='Random Baseline')
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

Understanding which features contribute most to the prediction is critical in medical contexts. We analyze feature importance from two perspectives:
- **Logistic Regression coefficients** — show direction and strength of each feature's effect
- **Random Forest feature importances** — show how much each feature contributes to tree splits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Logistic Regression coefficients
lr_importance = pd.Series(lr_model.coef_[0], index=X.columns).sort_values()
lr_importance.plot(kind='barh', ax=axes[0], color=lr_importance.apply(
    lambda x: '#e74c3c' if x > 0 else '#3498db'
))
axes[0].set_title('Logistic Regression — Coefficients', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Coefficient Value')
axes[0].axvline(x=0, color='black', linewidth=0.8)

# Random Forest feature importance
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()
rf_importance.plot(kind='barh', ax=axes[1], color='#2ecc71')
axes[1].set_title('Random Forest — Feature Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.suptitle('Feature Importance Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 9.1 Top Predictive Features

Let's summarize the most important features for predicting COVID-19 mortality.

In [ ]:
top_features = rf_importance.sort_values(ascending=False).head(5)
print('Top 5 features for predicting COVID-19 mortality (Random Forest):\n')
for i, (feature, importance) in enumerate(top_features.items(), 1):
    print(f'  {i}. {feature:<20s} — importance: {importance:.4f}')

## 10. Summary & Conclusions

### Key Findings

- Both models were trained with `class_weight='balanced'` to handle the ~7% death rate imbalance.
- **Recall** (catching actual death cases) was prioritized since this is a medical-risk classification.
- The feature importance analysis reveals which patient characteristics are strongest predictors of mortality.

### Limitations

- This is an **educational project** — the models are not validated for clinical use.
- Some columns with high missing rates (e.g., ICU, INTUBED) were dropped, which could carry useful signal.
- No hyperparameter tuning or cross-validation was performed (potential next step).
- The dataset may have inherent biases from the data collection process.

### Potential Next Steps

- Hyperparameter tuning with GridSearchCV or RandomizedSearchCV
- Cross-validation for more robust evaluation
- Try additional models (Gradient Boosting, XGBoost)
- Threshold tuning to optimize recall vs. precision trade-off
- SHAP values for more detailed feature interpretation